# 01 – Missing and Erroneous Values

**Objective:** identify and treat missing values and erroneous data in the Madrid property sales dataset, justifying every decision with evidence.

**Input:** `data/processed/listings_00.parquet` (output of notebook 00).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

# Consistent visual style
sns.set_theme(style="whitegrid", palette="colorblind", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

PERIOD_MAP = {"201803": "Q1 2018", "201806": "Q2 2018", "201809": "Q3 2018", "201812": "Q4 2018"}

try:
    df = pd.read_parquet("../data/processed/listings_00.parquet")
except FileNotFoundError:
    df = pd.read_csv("../data/Madrid_Sale.csv")

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 1. Missing values overview

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({"Missing": missing, "% of total": missing_pct})
display(missing_df)

# Visualise missing values
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: bar chart of missing percentages
severity_colors = ["#c0392c" if p > 50 else "#e67e22" if p > 5 else "#27ae60" for p in missing_pct]
bars = axes[0].barh(missing_pct.index[::-1], missing_pct.values[::-1], color=severity_colors[::-1], edgecolor="white")
axes[0].set_xlabel("Missing (%)")
axes[0].set_title("Missing values by column")
for i, (v, c) in enumerate(zip(missing_pct.values[::-1], missing.values[::-1])):
    axes[0].text(v + 0.8, i, f"{c:,} ({v}%)", va="center", fontsize=9)
axes[0].spines[["top", "right"]].set_visible(False)

# Right: heatmap of nullity for a sample
sample = df[missing.index].sample(500, random_state=42)
axes[1].imshow(sample.isnull().values.T, aspect="auto", cmap="Reds", interpolation="none")
axes[1].set_yticks(range(len(missing.index)))
axes[1].set_yticklabels(missing.index, fontsize=9)
axes[1].set_xlabel("Sample rows (n=500)")
axes[1].set_title("Nullity pattern (red = missing)")

plt.tight_layout()
plt.show()

Only 4 columns have missing values. `CONSTRUCTIONYEAR` dominates with 59% missing; the remaining three have modest gaps. The target (`PRICE`) and all binary features are 100% complete.

### 1.1 `CONSTRUCTIONYEAR` — 59% missing

This column is self-reported by the listing agent. It is missing for more than half the dataset and, as we show below, contains erroneous values. However, the cadastral alternative `CADCONSTRUCTIONYEAR` is 100% complete and comes from an official source (Spanish Cadastre).

In [ ]:
cy = df["CONSTRUCTIONYEAR"].dropna()
print(f"Non-null values: {len(cy):,} / {len(df):,} ({len(cy)/len(df):.1%})")
print(f"Range: {cy.min():.0f} – {cy.max():.0f}")

suspicious = cy[(cy < 1800) | (cy > 2020)]
print(f"\nSuspicious values (year < 1800 or > 2020): {len(suspicious)} rows")
print(f"Examples: {suspicious.head(10).tolist()}")

# Compare with cadastral year
valid = df[df["CONSTRUCTIONYEAR"].between(1800, 2020)]
corr = valid["CONSTRUCTIONYEAR"].corr(valid["CADCONSTRUCTIONYEAR"])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribution comparison
axes[0].hist(cy, bins=50, alpha=0.6, color="#e74c3c", label="Self-reported", edgecolor="white")
axes[0].hist(df["CADCONSTRUCTIONYEAR"], bins=50, alpha=0.6, color="#3498db", label="Cadastral (official)", edgecolor="white")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Number of listings")
axes[0].set_title("Construction year: self-reported vs cadastral")
axes[0].legend(frameon=True, fancybox=True)

# 2. Scatter: self-reported vs cadastral
scatter = axes[1].scatter(valid["CADCONSTRUCTIONYEAR"], valid["CONSTRUCTIONYEAR"],
                          c=valid["PRICE"], cmap="viridis", alpha=0.1, s=3,
                          norm=plt.Normalize(vmin=50_000, vmax=800_000))
axes[1].plot([1800, 2020], [1800, 2020], "r--", lw=2, label="Perfect agreement")
axes[1].set_xlabel("Cadastral year (official)")
axes[1].set_ylabel("Self-reported year")
axes[1].set_title(f"Agreement between sources (r = {corr:.3f})")
axes[1].legend(frameon=True)
plt.colorbar(scatter, ax=axes[1], label="Price (EUR)", shrink=0.8)

# 3. Zoom on erroneous values
axes[2].hist(cy[cy < 100], bins=30, color="#e74c3c", edgecolor="white")
axes[2].set_xlabel("Self-reported construction year")
axes[2].set_ylabel("Number of listings")
axes[2].set_title(f"Zoom: {len(cy[cy < 100])} erroneous entries (year < 100)")
axes[2].set_xlim(-5, 105)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print("\nSample rows with erroneous CONSTRUCTIONYEAR:")
err_rows = df[df["CONSTRUCTIONYEAR"].between(0, 100, inclusive="both")]
print(err_rows[["CONSTRUCTIONYEAR", "CADCONSTRUCTIONYEAR", "PRICE", "CONSTRUCTEDAREA"]].head(10))

**Decision:** drop `CONSTRUCTIONYEAR`. The cadastral source (`CADCONSTRUCTIONYEAR`) is complete, official, and highly correlated with the valid self-reported values. Keeping the self-reported column would force us to impute 59% of the data or lose those rows.

In [4]:
df = df.drop(columns=["CONSTRUCTIONYEAR"])
print(f"Dropped CONSTRUCTIONYEAR. Shape: {df.shape}")

Dropped CONSTRUCTIONYEAR. Shape: (94815, 40)


### 1.2 `FLATLOCATIONID` — 6.7% missing

This column indicates whether the flat is **interior (1)** or **exterior (2)**. Missing values likely mean the information was not provided in the listing.

In [5]:
print(df["FLATLOCATIONID"].value_counts(dropna=False))
print(f"\nMedian PRICE by FLATLOCATIONID:")
print(df.groupby("FLATLOCATIONID", dropna=False)["PRICE"].median())

FLATLOCATIONID
1.0    76324
2.0    12104
NaN     6387
Name: count, dtype: int64



Median PRICE by FLATLOCATIONID:
FLATLOCATIONID
1.0    280000.0
2.0    241000.0
NaN    173000.0
Name: PRICE, dtype: float64


Missing values have a distinct median price from both interior and exterior flats, confirming that "unknown" is a meaningful category, not random missingness.

**Decision:** fill `NaN` with 0 to represent "unknown location". This preserves the information that the value was not reported, rather than imputing a potentially wrong category.

In [6]:
df["FLATLOCATIONID"] = df["FLATLOCATIONID"].fillna(0).astype(int)
print(df["FLATLOCATIONID"].value_counts())

FLATLOCATIONID
1    76324
2    12104
0     6387
Name: count, dtype: int64


### 1.3 `FLOORCLEAN` — 4.1% missing

The floor number of the flat. The value -1 appears 936 times and likely represents basement or ground-level commercial units.

In [7]:
print(df["FLOORCLEAN"].value_counts(dropna=False).sort_index())
print(f"\nMedian PRICE by FLOORCLEAN (selected values):")
print(df.groupby("FLOORCLEAN", dropna=False)["PRICE"].median().sort_index())

FLOORCLEAN
-1.0       936
 0.0     10112
 1.0     20336
 2.0     17044
 3.0     14754
 4.0     11635
 5.0      6162
 6.0      3733
 7.0      2447
 8.0      1495
 9.0       814
 10.0      450
 11.0     1051
 NaN      3846
Name: count, dtype: int64

Median PRICE by FLOORCLEAN (selected values):
FLOORCLEAN
-1.0     195000.0
 0.0     176000.0
 1.0     256000.0
 2.0     257000.0
 3.0     254000.0
 4.0     266000.0
 5.0     355000.0
 6.0     419000.0
 7.0     437000.0
 8.0     438000.0
 9.0     336000.0
 10.0    379000.0
 11.0    423000.0
 NaN     238000.0
Name: PRICE, dtype: float64


Floor -1 has a lower median price than floor 0, consistent with basement or semi-basement properties. Missing values have a median price close to the overall median, suggesting no strong pattern.

**Decision:** keep -1 as a valid value (basement). Fill missing values with the median floor number.

In [8]:
median_floor = df["FLOORCLEAN"].median()
df["FLOORCLEAN"] = df["FLOORCLEAN"].fillna(median_floor)
print(f"Filled missing FLOORCLEAN with median = {median_floor}")

Filled missing FLOORCLEAN with median = 2.0


### 1.4 `CADASTRALQUALITYID` — 1 missing

In [9]:
mode_quality = df["CADASTRALQUALITYID"].mode()[0]
df["CADASTRALQUALITYID"] = df["CADASTRALQUALITYID"].fillna(mode_quality)
print(f"Filled 1 missing value with mode = {mode_quality}")

Filled 1 missing value with mode = 4.0


## 2. Erroneous values

### 2.1 Extreme `ROOMNUMBER` and `BATHNUMBER`

In [ ]:
extreme_rooms = df[df["ROOMNUMBER"] > 10]
print(f"Listings with ROOMNUMBER > 10: {len(extreme_rooms)}")
print(extreme_rooms[["ROOMNUMBER", "BATHNUMBER", "CONSTRUCTEDAREA", "PRICE"]].describe())

extreme_baths = df[df["BATHNUMBER"] > 5]
print(f"\nListings with BATHNUMBER > 5: {len(extreme_baths)}")
print(extreme_baths[["ROOMNUMBER", "BATHNUMBER", "CONSTRUCTEDAREA", "PRICE"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(df["CONSTRUCTEDAREA"], df["ROOMNUMBER"], alpha=0.03, s=2, color="#2c3e50")
axes[0].axhline(10, color="#e74c3c", linestyle="--", lw=1.5, label="Cap at 10")
axes[0].set_xlabel("Constructed area (m²)")
axes[0].set_ylabel("Number of rooms")
axes[0].set_title("Area vs rooms — values above 10 are errors")
axes[0].legend(frameon=True)

axes[1].scatter(df["CONSTRUCTEDAREA"], df["BATHNUMBER"], alpha=0.03, s=2, color="#2c3e50")
axes[1].axhline(5, color="#e74c3c", linestyle="--", lw=1.5, label="Cap at 5")
axes[1].set_xlabel("Constructed area (m²)")
axes[1].set_ylabel("Number of bathrooms")
axes[1].set_title("Area vs bathrooms — values above 5 are errors")
axes[1].legend(frameon=True)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

Most extreme room/bathroom counts occur in small flats (< 200 m²), which is implausible — a 60 m² flat cannot have 93 rooms. These are clearly data entry errors.

**Decision:** cap `ROOMNUMBER` at 10 and `BATHNUMBER` at 5. Values above these thresholds are inconsistent with the flat's area and represent errors, not luxury properties.

In [11]:
n_rooms_capped = (df["ROOMNUMBER"] > 10).sum()
n_baths_capped = (df["BATHNUMBER"] > 5).sum()
df.loc[df["ROOMNUMBER"] > 10, "ROOMNUMBER"] = 10
df.loc[df["BATHNUMBER"] > 5, "BATHNUMBER"] = 5
print(f"Capped ROOMNUMBER: {n_rooms_capped} rows")
print(f"Capped BATHNUMBER: {n_baths_capped} rows")

Capped ROOMNUMBER: 56 rows
Capped BATHNUMBER: 264 rows


### 2.2 `PARKINGSPACEPRICE` inconsistency

In [12]:
no_parking = df[df["HASPARKINGSPACE"] == 0]
print(f"Listings without parking: {len(no_parking)}")
print(f"PARKINGSPACEPRICE when no parking:")
print(no_parking["PARKINGSPACEPRICE"].value_counts().head())
print(f"\nMax PARKINGSPACEPRICE with no parking: {no_parking['PARKINGSPACEPRICE'].max()}")

Listings without parking: 73472
PARKINGSPACEPRICE when no parking:
PARKINGSPACEPRICE
1        71289
20001      230
30001      209
25001      185
15001      182
Name: count, dtype: int64

Max PARKINGSPACEPRICE with no parking: 925001


When `HASPARKINGSPACE = 0`, the parking price is mostly 1 (a placeholder). Some rows have large values, which are clearly errors since there is no parking space.

**Decision:** set `PARKINGSPACEPRICE = 0` when `HASPARKINGSPACE = 0`.

In [13]:
n_fixed = (df["HASPARKINGSPACE"] == 0).sum()
df.loc[df["HASPARKINGSPACE"] == 0, "PARKINGSPACEPRICE"] = 0
print(f"Set PARKINGSPACEPRICE = 0 for {n_fixed} listings without parking")

Set PARKINGSPACEPRICE = 0 for 73472 listings without parking


### 2.3 Geographic outlier

In [ ]:
geo_outlier = df[df["LATITUDE"] < 40]
print(f"Listings with LATITUDE < 40: {len(geo_outlier)}")
print(geo_outlier[["LATITUDE", "LONGITUDE", "PRICE", "CONSTRUCTEDAREA"]])

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df["LONGITUDE"], df["LATITUDE"], alpha=0.05, s=1, color="#3498db", label="Madrid listings")
ax.scatter(geo_outlier["LONGITUDE"], geo_outlier["LATITUDE"], c="red", s=80, zorder=5,
           edgecolors="darkred", linewidths=1.5, label="Outlier (Almería)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Geographic distribution — 1 listing outside Madrid")
ax.legend(frameon=True, markerscale=2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

One listing has coordinates in Almería (latitude 36.76°), far outside Madrid (latitude ~40.4°). This is a geocoding error.

**Decision:** remove this row.

In [15]:
df = df[df["LATITUDE"] >= 40]
print(f"Removed geographic outlier. Shape: {df.shape}")

Removed geographic outlier. Shape: (94814, 40)


### 2.4 Duplicate listings

In [16]:
n_total = len(df)
n_unique = df["ASSETID"].nunique()
print(f"Total rows: {n_total:,}")
print(f"Unique ASSETIDs: {n_unique:,}")
print(f"Duplicate rows: {n_total - n_unique:,}")

# Check if duplicates are exact (all columns identical)
dup_mask = df.duplicated(subset=df.columns.difference(["ASSETID"]), keep=False)
print(f"\nExact duplicate rows (all columns except ASSETID): {dup_mask.sum():,}")

# Check duplicates by ASSETID
dup_assets = df[df.duplicated(subset=["ASSETID"], keep=False)].sort_values("ASSETID")
print(f"\nSample duplicate ASSETID:")
sample_id = dup_assets["ASSETID"].iloc[0]
print(df[df["ASSETID"] == sample_id][["ASSETID", "PERIOD", "PRICE", "CONSTRUCTEDAREA", "ROOMNUMBER"]])

Total rows: 94,814
Unique ASSETIDs: 75,803
Duplicate rows: 19,011

Exact duplicate rows (all columns except ASSETID): 0



Sample duplicate ASSETID:
                     ASSETID  PERIOD     PRICE  CONSTRUCTEDAREA  ROOMNUMBER
1733   A10001362419935476012  201803  230000.0               28           1
21455  A10001362419935476012  201803  210000.0               28           1
66878  A10001362419935476012  201803  215000.0               28           1


Duplicate ASSETIDs represent the same property listed multiple times in the same quarter. Since no asset appears in more than one period, these are true duplicates (likely re-listings or scraping artifacts).

**Decision:** drop duplicates keeping the first occurrence.

In [17]:
n_before = len(df)
df = df.drop_duplicates(subset=["ASSETID"], keep="first")
print(f"Removed {n_before - len(df):,} duplicate ASSETIDs. Shape: {df.shape}")

Removed 19,011 duplicate ASSETIDs. Shape: (75803, 40)


### 2.5 Target leakage: `UNITPRICE`

In [18]:
diff = (df["UNITPRICE"] - df["PRICE"] / df["CONSTRUCTEDAREA"]).abs()
print(f"Max difference between UNITPRICE and PRICE/AREA: {diff.max():.2e}")
print("UNITPRICE is exactly PRICE / CONSTRUCTEDAREA — a direct function of the target.")

df = df.drop(columns=["UNITPRICE"])
print(f"\nDropped UNITPRICE. Shape: {df.shape}")

Max difference between UNITPRICE and PRICE/AREA: 6.37e-12
UNITPRICE is exactly PRICE / CONSTRUCTEDAREA — a direct function of the target.

Dropped UNITPRICE. Shape: (75803, 39)


## Summary

**Actions performed in this notebook:**

| Issue | Evidence | Action |
|---|---|---|
| `CONSTRUCTIONYEAR` 59% missing + erroneous values | Values like 1, 50, 2291; cadastral source is complete | **Dropped** (use `CADCONSTRUCTIONYEAR`) |
| `FLATLOCATIONID` 6.7% missing | Missing has distinct median price | Filled with 0 (unknown category) |
| `FLOORCLEAN` 4.1% missing | No strong pattern in missingness | Filled with median |
| `CADASTRALQUALITYID` 1 missing | Trivial | Filled with mode |
| `ROOMNUMBER` > 10 (56 rows) | Small flats with 93 rooms — data entry errors | Capped at 10 |
| `BATHNUMBER` > 5 (264 rows) | Small flats with 20 baths — data entry errors | Capped at 5 |
| `PARKINGSPACEPRICE` with no parking | Placeholder value 1 | Set to 0 |
| 1 row in Almería (lat 36.76°) | Geocoding error | **Removed** |
| ~19K duplicate ASSETIDs | Same property, same period | **Dropped duplicates** |
| `UNITPRICE` = PRICE / AREA | Direct target leakage | **Dropped** |

**Output:** saved to `data/processed/listings_01.parquet`.

In [19]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_parquet("../data/processed/listings_01.parquet", index=False)
print(f"Saved: {df.shape[0]} rows × {df.shape[1]} columns")

Saved: 75803 rows × 39 columns
